# 02 · Defect detection — YOLO11s

Trains and evaluates YOLO11s (Ultralytics) on EL images for four defect classes: *Black spot*, *Cell Scratch*, *Cells class difference* and *Micro-crack*. Includes the panel-crop + CLAHE + micro-crack oversampling preprocessing step used for the detector.

| | |
|---|---|
| **Input** | YOLO-format dataset + `data.yaml` (see `configs/data.example.yaml`) |
| **Output** | Training curves, validation metrics (mAP@50, precision, recall), exported model |
| **Runs on** | GPU (Google Colab, T4 used for the reported run) |

> **Notes**
> - `DATASET_PATH` points to the **raw** export. The preprocessing cell writes a processed copy to `detection_dataset_processed/`; point `DATASET_PATH` there to train on the preprocessed images.
> - Weights and datasets are not included in this repository (see `DATA.md`).

# YOLO11 Training on YOLO11 Format Data

This notebook trains YOLO11 (YOLOv11) model using YOLO format dataset from Google Drive.

**Requirements:**
- Dataset in YOLO11 format (images + labels)
- Dataset stored in Google Drive
- data.yaml configuration file

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Dependencies

In [ ]:
# Install Ultralytics (includes YOLO11)
!pip install ultralytics -q

# Verify installation
import ultralytics
ultralytics.checks()

print(f"\nUltralytics version: {ultralytics.__version__}")

## 3. Configuration

Update these paths to match your Google Drive structure:

In [ ]:
# ========== CONFIGURE THESE PATHS ==========

# Path to your dataset folder in Google Drive
DATASET_PATH = '/content/drive/MyDrive/inspectra/detection_dataset_raw'

# Path to data.yaml file (should be in your dataset folder)
DATA_YAML = f'{DATASET_PATH}/data.yaml'

# Where to save training results
OUTPUT_PATH = '/content/drive/MyDrive/inspectra/runs/yolo11'

# YOLO11 model size: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
MODEL_SIZE = 'yolo11s.pt'

# Training parameters
EPOCHS = 100
IMG_SIZE = 1280
BATCH_SIZE = 6 # 4 Adjust based on GPU memory

# ==========================================

## 4. Verify Dataset Structure

In [ ]:
import os
import yaml

# Check if dataset path exists
if not os.path.exists(DATASET_PATH):
    print(f"❌ Error: Dataset path not found: {DATASET_PATH}")
    print("Please update DATASET_PATH in the configuration cell above.")
else:
    print(f"✓ Dataset folder found: {DATASET_PATH}")

# Check data.yaml
if not os.path.exists(DATA_YAML):
    print(f"\n❌ Error: data.yaml not found: {DATA_YAML}")
else:
    print(f"\n✓ data.yaml found: {DATA_YAML}")

    # Display data.yaml contents
    with open(DATA_YAML, 'r') as f:
        data_config = yaml.safe_load(f)

    print("\n📋 Dataset Configuration:")
    print(f"  Classes: {data_config.get('nc', 'Not specified')}")
    print(f"  Names: {data_config.get('names', 'Not specified')}")
    print(f"  Train: {data_config.get('train', 'Not specified')}")
    print(f"  Val: {data_config.get('val', 'Not specified')}")

# List dataset structure
print("\n📂 Dataset Structure:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # Only show first 2 levels to avoid clutter
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")
        if len(files) > 5:
            print(f"{subindent}... and {len(files)-5} more files")

## 5. Check GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")

## 6. Initialize YOLO11 Model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11 model
print(f"Loading {MODEL_SIZE}...")
model = YOLO(MODEL_SIZE)

print("\n✓ Model loaded successfully!")
print(f"Model: {model.model_name}")

preprocessing


In [ ]:
# Install OpenCV
!pip install opencv-python

import cv2
import os
from pathlib import Path
import numpy as np


def find_crop_box(image_path, pad=5):
    """
    Detect solar panel area and remove black borders
    """

    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        return None

    # Blur to reduce noise
    blur = cv2.GaussianBlur(img, (7, 7), 0)

    # Automatic threshold using Otsu
    _, th = cv2.threshold(
        blur,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Invert if needed
    if np.mean(th) > 127:
        th = 255 - th

    # Morphological closing
    kernel = np.ones((15, 15), np.uint8)

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Erode for tighter crop
    th = cv2.erode(
        th,
        np.ones((5, 5), np.uint8),
        iterations=1
    )

    # Find contours
    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return None

    # Largest contour = panel
    largest = max(contours, key=cv2.contourArea)

    x, y, w, h = cv2.boundingRect(largest)

    H, W = img.shape

    # Padding
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)

    x2 = min(W, x + w + pad)
    y2 = min(H, y + h + pad)

    return x1, y1, x2, y2, W, H


def convert_yolo_label(line, old_W, old_H, crop):

    cls, xc, yc, bw, bh = line.strip().split()

    xc, yc, bw, bh = map(
        float,
        [xc, yc, bw, bh]
    )

    x1c, y1c, x2c, y2c = crop

    new_W = x2c - x1c
    new_H = y2c - y1c

    # YOLO normalized -> absolute
    x_center = xc * old_W
    y_center = yc * old_H

    box_w = bw * old_W
    box_h = bh * old_H

    x1 = x_center - box_w / 2
    y1 = y_center - box_h / 2

    x2 = x_center + box_w / 2
    y2 = y_center + box_h / 2

    # Shift after crop
    x1 -= x1c
    x2 -= x1c

    y1 -= y1c
    y2 -= y1c

    # Clip to the new image bounds
    x1 = max(0, min(new_W, x1))
    x2 = max(0, min(new_W, x2))

    y1 = max(0, min(new_H, y1))
    y2 = max(0, min(new_H, y2))

    # Ignore invalid boxes
    if x2 <= x1 or y2 <= y1:
        return None

    # Convert back to normalized
    new_xc = ((x1 + x2) / 2) / new_W
    new_yc = ((y1 + y2) / 2) / new_H

    new_bw = (x2 - x1) / new_W
    new_bh = (y2 - y1) / new_H

    return (
        f"{cls} "
        f"{new_xc:.6f} "
        f"{new_yc:.6f} "
        f"{new_bw:.6f} "
        f"{new_bh:.6f}"
    )


def process_dataset(
    images_dir,
    labels_dir,
    out_images_dir,
    out_labels_dir
):

    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)

    out_images_dir = Path(out_images_dir)
    out_labels_dir = Path(out_labels_dir)

    out_images_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    out_labels_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for img_path in images_dir.glob("*.jpg"):

        img = cv2.imread(str(img_path))

        if img is None:
            continue

        box = find_crop_box(img_path)

        if box is None:
            continue

        x1, y1, x2, y2, old_W, old_H = box

        # Crop image
        cropped = img[y1:y2, x1:x2]

        # Convert to grayscale
        gray = cv2.cvtColor(
            cropped,
            cv2.COLOR_BGR2GRAY
        )

        # CLAHE enhancement
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        enhanced = clahe.apply(gray)

        # Label paths
        label_path = labels_dir / f"{img_path.stem}.txt"

        if not label_path.exists():
            continue

        # Convert labels
        new_lines = []

        repeat = 1

        for line in label_path.read_text().splitlines():

            cls = int(line.split()[0])

            # Oversample Micro-crack class
            if cls == 3:
                repeat = 4

            converted = convert_yolo_label(
                line,
                old_W,
                old_H,
                crop=(x1, y1, x2, y2)
            )

            if converted:
                new_lines.append(converted)

        # Save repeated images + labels
        for i in range(repeat):

            new_image_name = (
                f"{img_path.stem}_{i}.jpg"
            )

            new_label_name = (
                f"{img_path.stem}_{i}.txt"
            )

            cv2.imwrite(
                str(out_images_dir / new_image_name),
                enhanced
            )

            (out_labels_dir / new_label_name).write_text(
                "\n".join(new_lines)
            )

    print("✓ Done: cropped images + updated labels + micro-crack oversampling")


# Run processing
process_dataset(
    images_dir="/content/drive/MyDrive/inspectra/detection_dataset_raw/train/images",

    labels_dir="/content/drive/MyDrive/inspectra/detection_dataset_raw/train/labels",

    out_images_dir="/content/drive/MyDrive/inspectra/detection_dataset_processed/train/images",

    out_labels_dir="/content/drive/MyDrive/inspectra/detection_dataset_processed/train/labels"
)


to know class id


In [ ]:
import yaml

yaml_path = "/content/drive/MyDrive/inspectra/detection_dataset_raw/data.yaml"

with open(yaml_path, "r") as file:
    data = yaml.safe_load(file)

print("names:")

names = data["names"]

# If names is a list
if isinstance(names, list):

    for i, name in enumerate(names):
        print(f"  {i}: {name}")

# If names is a dict
elif isinstance(names, dict):

    for key, value in names.items():
        print(f"  {key}: {value}")

## 7. Train the Model

This will start the training process. Training progress will be displayed below.

In [ ]:
# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Start training
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=OUTPUT_PATH,
    name='yolo11_training',
    device=0 if torch.cuda.is_available() else 'cpu',

    # Additional training arguments
    patience=30,  # Early stopping patience
    save=True,    # Save checkpoints
    save_period=10,  # Save checkpoint every 10 epochs

    # Data augmentation
    hsv_h=0.015,  # HSV-Hue augmentation
    hsv_s=0.7,    # HSV-Saturation augmentation

    #hsv_v=0.4,    # HSV-Value augmentation
    hsv_v=0.6,

    degrees=0.0,  # Rotation augmentation

    translate=0.1,  # Translation augmentation
    scale=0.2,    # Scale augmentation


    #shear=0.0,    # Shear augmentation
    #perspective=0.0,  # Perspective augmentation
    shear=2.0,
    perspective=0.0005,

    #flipud=0.0,   # Vertical flip probability
    flipud=0.5,

    fliplr=0.5,   # Horizontal flip probability

    mosaic=0.2,   # 0.5 Mosaic augmentation probability
   mixup=0.0,    # Mixup augmentation probability


    copy_paste=0.0,  # Copy-paste augmentation probability

    # Optimizer
    optimizer='auto',  # auto, SGD, Adam, AdamW, NAdam, RAdam, RMSProp
    lr0=0.01,     # Initial learning rate
    lrf=0.01,     # Final learning rate factor
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # Loss weights
    box=7.5,      # Box loss weight
    cls=0.5,      # Classification loss weight
    dfl=1.5,      # DFL loss weight

    # Other
    verbose=True,
    seed=0,
    deterministic=False,
    plots=True,   # Save training plots
    amp=True,     # Automatic Mixed Precision
    fraction=1.0, # Use 100% of dataset

    #close_mosaic=10,  # Disable mosaic augmentation in last N epochs
    close_mosaic=10, #15,
)

print("\n✓ Training completed!")

## 8. Display Training Results

In [ ]:
from IPython.display import Image, display
import glob

# Find the latest training folder
training_folders = glob.glob(f"{OUTPUT_PATH}/yolo11_training*")
if training_folders:
    latest_folder = max(training_folders, key=os.path.getctime)

    print(f"📊 Training results saved to: {latest_folder}\n")

    # Display training curves
    results_img = f"{latest_folder}/results.png"
    if os.path.exists(results_img):
        print("Training Curves:")
        display(Image(filename=results_img, width=1000))

    # Display confusion matrix
    confusion_matrix = f"{latest_folder}/confusion_matrix.png"
    if os.path.exists(confusion_matrix):
        print("\nConfusion Matrix:")
        display(Image(filename=confusion_matrix, width=800))

    # Display normalized confusion matrix
    confusion_matrix_norm = f"{latest_folder}/confusion_matrix_normalized.png"
    if os.path.exists(confusion_matrix_norm):
        print("\nNormalized Confusion Matrix:")
        display(Image(filename=confusion_matrix_norm, width=800))

    # Display sample predictions
    val_batch = f"{latest_folder}/val_batch0_pred.jpg"
    if os.path.exists(val_batch):
        print("\nSample Validation Predictions:")
        display(Image(filename=val_batch, width=1000))

    # Display F1 curve
    f1_curve = f"{latest_folder}/F1_curve.png"
    if os.path.exists(f1_curve):
        print("\nF1 Confidence Curve:")
        display(Image(filename=f1_curve, width=800))

    # Display PR curve
    pr_curve = f"{latest_folder}/PR_curve.png"
    if os.path.exists(pr_curve):
        print("\nPrecision-Recall Curve:")
        display(Image(filename=pr_curve, width=800))
else:
    print("No training results found.")

## 9. Validate the Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print("\n📈 Validation Metrics:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")
print(f"  F1 Score: {2 * (metrics.box.mp * metrics.box.mr) / (metrics.box.mp + metrics.box.mr + 1e-6):.4f}")

## 10. Export the Model

In [ ]:
# Export to different formats
export_folder = f"{OUTPUT_PATH}/exported_models"
os.makedirs(export_folder, exist_ok=True)

print("Exporting model...\n")

# Export to ONNX (recommended for deployment)
try:
    onnx_path = model.export(format='onnx', dynamic=True, simplify=True)
    print(f"✓ ONNX model exported: {onnx_path}")
except Exception as e:
    print(f"❌ ONNX export failed: {e}")

# Export to TorchScript
try:
    torchscript_path = model.export(format='torchscript')
    print(f"✓ TorchScript model exported: {torchscript_path}")
except Exception as e:
    print(f"❌ TorchScript export failed: {e}")

# Export to TensorFlow Lite (for mobile)
try:
    tflite_path = model.export(format='tflite')
    print(f"✓ TFLite model exported: {tflite_path}")
except Exception as e:
    print(f"❌ TFLite export failed: {e}")

print(f"\nBest weights saved at: {latest_folder}/weights/best.pt")

## 11. Test Inference on Sample Images

In [ ]:
# Load the best model
best_model = YOLO(f"{latest_folder}/weights/best.pt")

# Run inference on validation images
test_results = best_model.predict(
    source="/content/drive/MyDrive/inspectra/detection_dataset_raw/valid/images",  # Adjust path as needed
    conf=0.25,  # Confidence threshold
    iou=0.7,    # NMS IOU threshold
    save=True,
    save_txt=True,
    save_conf=True,
    project=f"{OUTPUT_PATH}/predictions",
    name='test_predictions',
    max_det=300,  # Maximum detections per image
    line_width=2,
    show_labels=True,
    show_conf=True,
)

print("\n✓ Predictions saved!")

# Display sample predictions
pred_images = glob.glob(f"{OUTPUT_PATH}/predictions/test_predictions/*.jpg")
if pred_images:
    print("\nSample Predictions:")
    for img_path in pred_images[:3]:  # Show first 3 predictions
        display(Image(filename=img_path, width=800))
        print()
else:
    print("No prediction images found.")

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\n✓ Model: YOLO11 {MODEL_SIZE}")
print(f"✓ Epochs completed: {EPOCHS}")
print(f"✓ Final mAP50-95: {metrics.box.map:.4f}")
print(f"✓ Final mAP50: {metrics.box.map50:.4f}")
print(f"✓ Precision: {metrics.box.mp:.4f}")
print(f"✓ Recall: {metrics.box.mr:.4f}")
print(f"\n📁 Results saved to: {OUTPUT_PATH}")
print(f"📁 Best weights: {latest_folder}/weights/best.pt")
print(f"📁 Last weights: {latest_folder}/weights/last.pt")

## 12. Advanced: Hyperparameter Tuning (Optional)

In [ ]:
# Uncomment to run hyperparameter tuning
# WARNING: This will take a long time!

# model = YOLO(MODEL_SIZE)
# tuning_results = model.tune(
#     data=DATA_YAML,
#     epochs=30,
#     iterations=300,
#     optimizer='AdamW',
#     plots=True,
#     save=True,
#     val=True,
# )

print("Hyperparameter tuning is disabled by default.")
print("Uncomment the code above to enable it.")

## 13. Benchmark Speed (Optional)

In [ ]:
# Benchmark model speed
print("Benchmarking model speed...\n")

benchmark_results = best_model.benchmark(
    imgsz=IMG_SIZE,
    half=True,  # FP16 precision
    device=0 if torch.cuda.is_available() else 'cpu',
)

print("\n✓ Benchmark completed!")

## 14. Summary & Next Steps

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\n✓ Model: YOLO11 {MODEL_SIZE}")
print(f"✓ Epochs completed: {EPOCHS}")
print(f"✓ Final mAP50-95: {metrics.box.map:.4f}")
print(f"✓ Final mAP50: {metrics.box.map50:.4f}")
print(f"✓ Precision: {metrics.box.mp:.4f}")
print(f"✓ Recall: {metrics.box.mr:.4f}")
print(f"\n📁 Results saved to: {OUTPUT_PATH}")
print(f"📁 Best weights: {latest_folder}/weights/best.pt")
print(f"📁 Last weights: {latest_folder}/weights/last.pt")

print("\n" + "="*60)
print("NEXT STEPS")
print("="*60)
print("1. Review training curves and metrics above")
print("2. Test on your own images using the inference section")
print("3. Export model for deployment (ONNX, TFLite, etc.)")
print("4. Download weights from Google Drive for local use")
print("5. Fine-tune hyperparameters if needed and retrain")
print("6. Try different model sizes (yolo11n, s, m, l, x)")
print("\n" + "="*60)